# Path-Orphan — Colab runner
Detect **conditional / orphan essentials** (essential genes with conservation<0.1) using dN/dS + AFDB/Foldseek structural homology + cofitness, on top of conservation, in *Ralstonia solanacearum* GMI1000.

**The bar (measured):** conservation R@P30 in the rogue zone = **0.000**. Conservation has zero purchase here, so the gate is absolute: the combined model must produce *any* P>=0.30 head (>=3.6× base rate) that beats the permutation null.

**Phase split — GPU on for ONE step only:**
| phase | runtime | steps |
|---|---|---|
| **A** | CPU (free) | 0 bridge ✓, 2 baseline ✓, 1 CES, 3 dN/dS, 4 Foldseek, 5 cofit, 7 DEG |
| **B** | A100/L4 | ESM-2 embeddings (only this) |
| **C** | CPU | 6 model+gate, 8 null |

Every step writes to `outputs/orphan/` **and** Drive, and is **skip-if-exists** — a runtime switch never loses work. STATUS legend: ✅ runs now · ⏳ needs feba.db/tools/all-org bridges (wired, staged).

## Setup (CPU runtime) — clone, mount Drive, copy feba.db

In [ ]:
import subprocess, os
from pathlib import Path
REPO = Path('/content/cell'); BRANCH='claude/vectorize-gex-propensity-NRqBW'
if not REPO.exists():
    subprocess.run(['git','clone','-b',BRANCH,'https://github.com/Nikku03/cell.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','origin',BRANCH],check=True)
os.chdir(REPO)
from google.colab import drive; drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/path_orphan'); DRIVE.mkdir(parents=True,exist_ok=True)
# feba.db (7.3 GB) needed by steps 1 (CES) and 5 (cofit)
FEBA_SRC = Path('/content/drive/MyDrive/cell_count_dynamics/multiorg/fitness_browser/feba.db')
if FEBA_SRC.exists() and not Path('/content/feba.db').exists():
    subprocess.run(['cp',str(FEBA_SRC),'/content/feba.db'],check=True)
print('feba.db present:', Path('/content/feba.db').exists())
subprocess.run(['pip','-q','install','pandas','pyarrow','scikit-learn','xgboost','biopython'],check=True)

## Phase A — ✅ Step 0: locus-tag bridge (cached)

In [ ]:
!python scripts/orphan_bridge.py --real
!cp -n outputs/orphan/bridge_* outputs/orphan/proteins_* outputs/orphan/uniprot_request_* {str(DRIVE)}/ 2>/dev/null; echo done

## Phase A — ✅ Step 2: conservation baseline = THE BAR

In [ ]:
!python scripts/orphan_baseline.py --real
!cp outputs/orphan/baseline_* {str(DRIVE)}/

## Phase A — ⏳ Step 3: dN/dS (Nei-Gojobori)
Core NG math is built+tested (`--smoke`). Real mode needs nucleotide CDS for each OG's members. Two prep sub-steps:
1. **all-org bridges + OG→protein_id map** (so orthologs can be resolved to sequences).
2. **download cds_from_genomic** for the assemblies (NCBI datasets), into `outputs/orphan/cds/`.
Then the alignment+NG driver runs on CPU.

In [ ]:
!python scripts/orphan_dnds.py --smoke   # validates the NG core now
# --- real prep (STAGED) ---
# 1) build bridges for all assemblies -> OG->protein_id map (orphan_dnds real path consumes it)
# 2) download CDS nucleotide fastas:
#    !pip -q install ncbi-datasets-cli
#    for acc in <assemblies>: datasets download genome accession $acc --include cds
#    unzip + concat *_cds_from_genomic.fna -> outputs/orphan/cds/<acc>.fna
# 3) !python scripts/orphan_dnds.py --real --cds_dir outputs/orphan/cds
print('dN/dS core OK; real prep staged above')

## Phase A — ⏳ Step 4: AFDB structures + Foldseek
Map RefSeq protein_ids → UniProt (from `uniprot_request_*.txt`), pull AFDB PDBs, Foldseek vs **AFDB-cluster reps** (~25 GB, once). Name-grade hits (STRICT=characterized fold vs LOOSE=orphan↔orphan). CPU; pLDDT>=70 filter.

In [ ]:
# install foldseek (static build)
# !wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz && tar xzf foldseek-linux-avx2.tar.gz
# UniProt idmapping: RefSeq Protein -> UniProtKB for ids in uniprot_request_*.txt
# AFDB pull: for each UniProt acc -> https://alphafold.ebi.ac.uk/files/AF-<acc>-F1-model_v4.pdb
# foldseek easy-search queries/ afdb_clust_db result.m8 tmp --format-output 'query,target,fident,evalue,bits'
# -> scripts/orphan_foldseek.py --real  (STATUS: building)
print('Step 4 staged')

## Phase A — ⏳ Step 5: cofitness feature (feba.db `Cofit`)

In [ ]:
# median essentiality of each gene's top-10 cofit partners (feba.db Cofit table)
# -> scripts/orphan_cofit.py --real --feba /content/feba.db  (STATUS: building)
print('Step 5 staged (needs feba.db)')

## Phase A — ⏳ Step 1: CES consensus label (feba.db) + Step 7: DEG1057 cross-grade

In [ ]:
# Step 1: reuse ces_consensus.py logic keyed to GMI1000 -> y_ess + total_weight
# !python scripts/ces_consensus.py --real --feba_db /content/feba.db --no_download
# Step 7: grade rogue-zone wins vs DEG1057 (independent screen) -- data/drive_import/deg
# -> scripts/orphan_deg_grade.py --real  (STATUS: building)
print('Steps 1 + 7 staged')

## ⚠️ SWITCH RUNTIME → A100 (or L4) now — Phase B only
Runtime → Change runtime type → GPU. Re-run the **Setup** cell (VM reset; Drive persists), then run only the cell below.

In [ ]:
# ESM-2 650M on outputs/orphan/proteins_*.faa -> esm_*.parquet (~20 min A100)
# reuse scripts/build_esm2_embeddings.py; key by locus_tag
# -> scripts/orphan_esm.py --real  (STATUS: building)
# !cp outputs/orphan/esm_* {DRIVE}/
print('Phase B (ESM) staged — GPU')

## ⚠️ SWITCH RUNTIME → CPU now — Phase C

In [ ]:
# Step 6: model = family_frac + dN/dS + fold_hit_named + cofit + ESM, leak-free
#         report rogue-zone R@P30 vs the bar (0.0). -> scripts/orphan_model.py --real
# Step 8: permutation null (shuffle y_ess, fixed features, 100x). -> scripts/orphan_null.py --real
# GATE: rogue-zone R@P30 > 0 AND > permutation-null R@P30 -> orphans tractable.
print('Phase C (model + null) staged')